# DataSage 4/4 — Aggregate Results & Visualize

Combines outputs from the 3 inference notebooks + W&B export into a single file for the demo dashboard.

| Setting | Value |
|---------|-------|
| Runtime | CPU (no GPU needed) |
| Input files | `wandb_training_data.json`, `cleaning_results.json`, `enrichment_results.json`, `answering_results.json` |
| Output | `evaluation_results.json`, PNG charts |

**How to use in Colab:** Run cells in order. Cell 2 will prompt you to upload the 4 JSON files from the previous notebooks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/04_aggregate_results.ipynb)

In [ ]:
# Cell 1: Install dependencies
!pip install -q matplotlib numpy

In [ ]:
# Cell 2: Upload input files (Colab) or read from disk (local)
import os
import sys
import json
import numpy as np
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

REQUIRED_FILES = [
    "wandb_training_data.json",
    "cleaning_results.json",
    "enrichment_results.json",
    "answering_results.json",
]

if IN_COLAB:
    # Check which files are missing
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
    if missing:
        from google.colab import files as colab_files
        print(f"Missing files: {missing}")
        print("Please upload them now:")
        uploaded = colab_files.upload()
        print(f"Uploaded: {list(uploaded.keys())}")
    else:
        print("All files already present.")
else:
    print("Running locally — reading from current directory.")

# Load all files
data = {}
for fname in REQUIRED_FILES:
    if os.path.exists(fname):
        with open(fname) as f:
            data[fname] = json.load(f)
        print(f"  Loaded: {fname} ({os.path.getsize(fname)/1024:.1f} KB)")
    else:
        print(f"  MISSING: {fname} — will use empty dict")
        data[fname] = {}

wandb_data = data["wandb_training_data.json"]
cleaning   = data["cleaning_results.json"]
enrichment = data["enrichment_results.json"]
answering  = data["answering_results.json"]

In [ ]:
# Cell 3: Build combined evaluation_results.json
import os
import json
from datetime import datetime

evaluation_results = {
    "datasage_finetuned": {
        "cleaning":   cleaning.get("datasage_lora", {}),
        "enrichment": enrichment.get("datasage_lora", {}),
        "answering":  answering.get("datasage_lora", {}),
    },
    "base_model": {
        "model":      "unsloth/Qwen2.5-3B-Instruct",
        "cleaning":   cleaning.get("base_model", {}),
        "enrichment": enrichment.get("base_model", {}),
        "answering":  answering.get("base_model", {}),
    },
    "wandb_training": wandb_data,
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "base_model": "unsloth/Qwen2.5-3B-Instruct",
        "lora_repos": {
            "cleaning":   "ricalanis/cleaning-grpo",
            "enrichment": "ricalanis/enrichment-grpo",
            "answering":  "ricalanis/answering-grpo",
        },
        "environments": {
            "cleaning":   "https://ricalanis-datasage-cleaning.hf.space",
            "enrichment": "https://ricalanis-datasage-enrichment.hf.space",
            "answering":  "https://ricalanis-datasage-answering.hf.space",
        },
        "domains":    ["hr", "sales", "pm", "it_ops"],
        "personas":   ["executive", "manager", "ic"],
        "wandb_runs": {
            "cleaning":   "xuwyjpe6",
            "enrichment": "orww3s2q",
            "answering":  "2mltqk5w",
        },
    },
}

OUTPUT_FILE = "evaluation_results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(evaluation_results, f, indent=2)

print(f"Saved: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024:.1f} KB)")

In [ ]:
# Cell 4: Print comparison table
import numpy as np

ft   = evaluation_results["datasage_finetuned"]
base = evaluation_results["base_model"]

print("\n" + "=" * 74)
print("DATASAGE EVALUATION RESULTS")
print("=" * 74)
print(f"\n{'Task':<15} {'Metric':<25} {'DataSage':>12} {'Base':>12} {'Delta':>10}")
print("-" * 74)

# Cleaning
fc = ft.get("cleaning", {})
bc = base.get("cleaning", {})
if fc and bc and "error" not in fc and "error" not in bc:
    print(f"{'Cleaning':<15} {'Final DQ':<25} {fc.get('final_dq_mean',0):>12.4f} {bc.get('final_dq_mean',0):>12.4f} {fc.get('final_dq_mean',0)-bc.get('final_dq_mean',0):>+10.4f}")
    print(f"{'':.<15} {'Avg Reward':<25} {fc.get('avg_reward_mean',0):>12.4f} {bc.get('avg_reward_mean',0):>12.4f} {fc.get('avg_reward_mean',0)-bc.get('avg_reward_mean',0):>+10.4f}")

# Enrichment
fe = ft.get("enrichment", {})
be = base.get("enrichment", {})
if fe and be and "error" not in fe and "error" not in be:
    print(f"{'Enrichment':<15} {'Coverage':<25} {fe.get('final_coverage_mean',0):>12.4f} {be.get('final_coverage_mean',0):>12.4f} {fe.get('final_coverage_mean',0)-be.get('final_coverage_mean',0):>+10.4f}")
    print(f"{'':.<15} {'Avg Reward':<25} {fe.get('avg_reward_mean',0):>12.4f} {be.get('avg_reward_mean',0):>12.4f} {fe.get('avg_reward_mean',0)-be.get('avg_reward_mean',0):>+10.4f}")

# Answering
fa = ft.get("answering", {})
ba = base.get("answering", {})
if fa and ba and "error" not in fa and "error" not in ba:
    print(f"{'Answering':<15} {'Reward':<25} {fa.get('reward_mean',0):>12.4f} {ba.get('reward_mean',0):>12.4f} {fa.get('reward_mean',0)-ba.get('reward_mean',0):>+10.4f}")
    print(f"{'':.<15} {'Faithfulness':<25} {fa.get('faithfulness_mean',0):>12.4f} {ba.get('faithfulness_mean',0):>12.4f} {fa.get('faithfulness_mean',0)-ba.get('faithfulness_mean',0):>+10.4f}")

# E2E composite score
print("-" * 74)
def e2e(r):
    c = r.get("cleaning", {}).get("avg_reward_mean", 0)
    e = r.get("enrichment", {}).get("avg_reward_mean", 0)
    a = r.get("answering", {}).get("reward_mean", 0)
    return 0.3 * c + 0.3 * e + 0.4 * a

ft_e2e = e2e(ft)
b_e2e  = e2e(base)
print(f"{'E2E Score':<15} {'0.3C+0.3E+0.4A':<25} {ft_e2e:>12.4f} {b_e2e:>12.4f} {ft_e2e-b_e2e:>+10.4f}")

In [ ]:
# Cell 5: W&B training reward curves
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for Colab
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("GRPO Training Reward Curves (from W&B)", fontsize=14, fontweight="bold")

for idx, (task, wd) in enumerate(wandb_data.items()):
    ax = axes[idx]
    ax.set_title(f"{task.title()} (run: {wd.get('run_id', '?')})")

    if "error" in wd:
        ax.text(0.5, 0.5, f"Error: {wd['error']}", ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color="red")
        continue

    metrics = wd.get("metrics", {})
    # Find reward or loss keys with multiple data points
    keys = sorted([k for k in metrics if "reward" in k.lower() and metrics[k]["n_points"] > 1])
    if not keys:
        keys = sorted([k for k in metrics if "loss" in k.lower() and metrics[k]["n_points"] > 1])

    for key in keys[:4]:
        label = key.split("/")[-1]  # strip prefix like "train/"
        ax.plot(metrics[key]["values"], label=label, alpha=0.8)

    ax.set_xlabel("Step")
    ax.set_ylabel("Value")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("wandb_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: wandb_training_curves.png")

In [ ]:
# Cell 6: Comparison bar chart — DataSage vs Base
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("DataSage GRPO vs Base Qwen2.5-3B", fontsize=14, fontweight="bold")

tasks_config = [
    ("Cleaning",   "avg_reward_mean",      "#F59E0B"),
    ("Enrichment", "final_coverage_mean",   "#EF4444"),
    ("Answering",  "reward_mean",           "#3B82F6"),
]

for i, (title, key, color) in enumerate(tasks_config):
    ax = axes[i]
    task_lower = title.lower()
    ft_val = ft.get(task_lower, {}).get(key, 0)
    b_val  = base.get(task_lower, {}).get(key, 0)

    bars = ax.bar(["DataSage", "Base"], [ft_val, b_val], color=[color, "#9CA3AF"])
    ax.set_title(f"{title} ({key.replace('_mean', '')})")
    ax.set_ylim(0, 1)

    for bar, val in zip(bars, [ft_val, b_val]):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("datasage_vs_base_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: datasage_vs_base_comparison.png")

In [ ]:
# Cell 7: Download all output files
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

output_files = [
    "evaluation_results.json",
    "wandb_training_curves.png",
    "datasage_vs_base_comparison.png",
]

print("Output files:")
for f in output_files:
    if os.path.exists(f):
        print(f"  {f}: {os.path.getsize(f)/1024:.1f} KB")
    else:
        print(f"  {f}: NOT FOUND")

if IN_COLAB:
    from google.colab import files as colab_files
    for f in output_files:
        if os.path.exists(f):
            colab_files.download(f)
    print("\nDownloaded to your browser.")
else:
    print("\nCopy evaluation_results.json to demo/data/ for the dashboard.")

print("\nDone!")